# Experiment 6: Pie and Donut Charts for Category Proportion

**Aim:** Show the share of total sales contributed by each product category using pie and donut charts.

**Dataset note:** The uploaded files are a retail orders dataset, not an Olympic medals dataset. Therefore this experiment uses category-wise sales proportions; medal share cannot be calculated from these files.

## Import libraries and load the supplied CSV files
The five dimension/fact tables are joined using their ID/date keys. Sales, cost and profit are calculated from the supplied unit price, unit cost, units sold and discount.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

DATA_DIR = Path("datasets")

def load_data():
    required_files = [
        "Dim_Customer.csv", "Dim_Date.csv", "Dim_Geography.csv",
        "Dim_Product.csv", "Fact_Orders.csv"
    ]
    missing = [f for f in required_files if not (DATA_DIR / f).exists()]
    if missing:
        raise FileNotFoundError(
            "Missing CSV file(s): " + ", ".join(missing) +
            ". Put all five CSV files in the datasets folder beside this notebook."
        )
    customer = pd.read_csv(DATA_DIR / "Dim_Customer.csv")
    dates = pd.read_csv(DATA_DIR / "Dim_Date.csv")
    geography = pd.read_csv(DATA_DIR / "Dim_Geography.csv")
    product = pd.read_csv(DATA_DIR / "Dim_Product.csv")
    orders = pd.read_csv(DATA_DIR / "Fact_Orders.csv")

    # Join the fact table with dimension tables using their matching keys.
    data = orders.merge(customer, on="Customer_ID", how="left")
    data = data.merge(product, on="Product_ID", how="left")
    data = data.merge(geography, on="Geo_ID", how="left")
    data = data.merge(dates, left_on="Order_Date", right_on="Date", how="left")

    data["Order_Date"] = pd.to_datetime(data["Order_Date"], errors="coerce")
    data["Ship_Date"] = pd.to_datetime(data["Ship_Date"], errors="coerce")
    data["Sales"] = data["Units_Sold"] * data["Unit_Price"] * (1 - data["Discount_Pct"])
    data["Cost"] = data["Units_Sold"] * data["Unit_Cost"]
    data["Profit"] = data["Sales"] - data["Cost"]
    data["Profit_Margin"] = np.where(data["Sales"] != 0,
                                     data["Profit"] / data["Sales"] * 100, 0)
    data["Order_Month"] = data["Order_Date"].dt.to_period("M").astype(str)
    return data

## Python code and visualization

In [ ]:
df = load_data()
category_sales = df.groupby("Category")["Sales"].sum().sort_values(ascending=False)

# Pie chart.
plt.figure(figsize=(7, 7))
plt.pie(category_sales.values, labels=category_sales.index,
        autopct="%1.1f%%", startangle=90)
plt.title("Category-wise Sales Share")
plt.axis("equal")
plt.show()

# Donut chart.
plt.figure(figsize=(7, 7))
plt.pie(category_sales.values, labels=category_sales.index,
        autopct="%1.1f%%", startangle=90,
        wedgeprops={"width": 0.42})
plt.title("Category-wise Sales Distribution – Donut Chart")
plt.axis("equal")
plt.show()

## Result
The experiment was performed using the supplied retail dataset. Run all cells to generate the tables, calculations and visualizations.